# Demonstração do Assistente Médico no Colab

Este notebook prepara somente a inferência e a interface. Ele não executa novamente o fine-tuning. Antes de começar, selecione uma GPU T4 em **Ambiente de execução > Alterar o tipo de ambiente de execução**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baixar ou atualizar o projeto

In [ ]:
%cd /content
!if [ -d tech-challenge-fase-3-assistente-medico/.git ]; then git -C tech-challenge-fase-3-assistente-medico pull --ff-only; else git clone https://github.com/camilasflores/tech-challenge-fase-3-assistente-medico.git; fi
%cd /content/tech-challenge-fase-3-assistente-medico

## 2. Instalar as dependências

A remoção do TorchAO evita o conflito conhecido entre a versão opcional pré-instalada no Colab e o PEFT.

In [ ]:
!pip uninstall -y torchao
!pip install -q -r requirements.txt

## 3. Baixar e preparar o adaptador LoRA

In [ ]:
!wget -q --show-progress -O /content/assistente-medico-lora-inferencia.zip https://github.com/camilasflores/tech-challenge-fase-3-assistente-medico/releases/download/v1.0.0/assistente-medico-lora-inferencia.zip
!python -m app.models.adapter_archive /content/assistente-medico-lora-inferencia.zip /content/assistente-medico-lora
%env LORA_ADAPTER_PATH=/content/assistente-medico-lora

## 4. Criar a base SQLite

In [ ]:
!python -m app.database.seed

## 5. Iniciar a interface Streamlit

As opções de CORS e XSRF abaixo são usadas somente no proxy temporário e autenticado do Colab.

In [ ]:
!pkill -f '[s]treamlit run streamlit_app.py' || true
!nohup python -m streamlit run streamlit_app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false --browser.gatherUsageStats false > /content/streamlit.log 2>&1 &

## 6. Verificar o servidor e abrir a aplicação

Aguarde o resultado `ok` e abra o endereço exibido em uma nova aba. Na primeira consulta, os modelos serão baixados e carregados; faça esse aquecimento antes de iniciar a gravação.

In [ ]:
import time
from google.colab import output

time.sleep(8)
!curl -s http://localhost:8501/_stcore/health
url = output.eval_js("google.colab.kernel.proxyPort(8501)")
print("Abra a interface:", url)